## Data Gathering

In [1]:
import pandas as pd

In [2]:
data=pd.read_csv('huge_1M_titanic.csv')

## Data Preprocessing

In [3]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1310,1,1,"Name1310, Miss. Surname1310",female,NaN,0,0,SOTON/O2 3101272,76.760165,NaN,C
1,1311,0,3,"Name1311, Col. Surname1311",male,29.0,0,0,223596,10.193097,NaN,S
2,1312,0,3,"Name1312, Mr. Surname1312",male,20.0,0,0,54636,12.029416,C83,C
3,1313,0,3,"Name1313, Mr. Surname1313",male,27.0,0,0,PC 17760,13.429448,NaN,S
4,1314,0,3,"Name1314, Mr. Surname1314",male,32.0,0,0,364512,4.840769,E33,C


Survived is the dependent column and rest all are independent column(input)

In [4]:
data.shape

(1000000, 12)

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 12 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   PassengerId  1000000 non-null  int64  
 1   Survived     1000000 non-null  int64  
 2   Pclass       1000000 non-null  int64  
 3   Name         1000000 non-null  str    
 4   Sex          1000000 non-null  str    
 5   Age          801400 non-null   float64
 6   SibSp        1000000 non-null  int64  
 7   Parch        1000000 non-null  int64  
 8   Ticket       1000000 non-null  str    
 9   Fare         1000000 non-null  float64
 10  Cabin        229805 non-null   str    
 11  Embarked     997760 non-null   str    
dtypes: float64(2), int64(5), str(5)
memory usage: 132.6 MB


In [6]:
data.isnull().sum()

PassengerId         0
Survived            0
Pclass              0
Name                0
Sex                 0
Age            198600
SibSp               0
Parch               0
Ticket              0
Fare                0
Cabin          770195
Embarked         2240
dtype: int64

Total 2240 missing value for embark,less data, won't affect much so we can remove those missing values, in cabin around 80% data is missing so will impact so either remove that column or fill with avg values(here we are directly removing it because filling with average values of less data will not give proper values of input), with age column we can go with filling missing values with avg values(here we are not using age as well, we should use but here just remove it).

In [7]:
# remove values that doesn't contribute in the prediction of survival or with more number of missing values
data=data.drop(['PassengerId','Name','Age','Ticket','Cabin'],axis=1)

In [8]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,C
1,0,3,male,0,0,10.193097,S
2,0,3,male,0,0,12.029416,C
3,0,3,male,0,0,13.429448,S
4,0,3,male,0,0,4.840769,C


In [9]:
data.isnull().sum()

Survived       0
Pclass         0
Sex            0
SibSp          0
Parch          0
Fare           0
Embarked    2240
dtype: int64

In [10]:
data['Embarked'].mode() #for categorical column go with mode, for numerical go with mean. Maximum people have embarked from S.

0    S
Name: Embarked, dtype: str

In [11]:
data['Embarked'].value_counts()

Embarked
S    729468
C    186846
Q     81446
Name: count, dtype: int64

## Data Cleaning

In [12]:
data['Embarked']=data['Embarked'].replace({'S':'Southampton','C':'Cherbourg','Q':'Queenstown'})

In [13]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,Cherbourg
1,0,3,male,0,0,10.193097,Southampton
2,0,3,male,0,0,12.029416,Cherbourg
3,0,3,male,0,0,13.429448,Southampton
4,0,3,male,0,0,4.840769,Cherbourg


In [14]:
data['Embarked'].value_counts()

Embarked
Southampton    729468
Cherbourg      186846
Queenstown      81446
Name: count, dtype: int64

In [15]:
data.dropna(subset=['Embarked'],inplace=True) #removing missing values of Embarked column

In [16]:
data.isnull().sum()

Survived    0
Pclass      0
Sex         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [17]:
data=data.reset_index(drop=True) #resetting index after dropping rows with missing values

In [18]:
data.shape

(997760, 7)

In [ ]:
data['Fare']=data['Fare'].astype(int) #large decimals were not required so converting to int

In [20]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76,Cherbourg
1,0,3,male,0,0,10,Southampton
2,0,3,male,0,0,12,Cherbourg
3,0,3,male,0,0,13,Southampton
4,0,3,male,0,0,4,Cherbourg


## Feature Encoding

convert categorical into numerical columns. for Gender--->LabelEncoder(used for binary data, yes/no)

In [21]:
from sklearn.preprocessing import LabelEncoder

In [22]:
label=LabelEncoder()

In [23]:
data['Sex']=label.fit_transform(data['Sex'])

In [24]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,0,0,0,76,Cherbourg
1,0,3,1,0,0,10,Southampton
2,0,3,1,0,0,12,Cherbourg
3,0,3,1,0,0,13,Southampton
4,0,3,1,0,0,4,Cherbourg


Ordinal Encoders are used when categories have some hierarchial order or relation like Mtech>Btech>12th, so we assign priority values to them like 3>2>1, but in embarked column there is no order or relation, all three categories are completely unrelated so we use one hot encoding.

In [25]:
from sklearn.preprocessing import OneHotEncoder

In [26]:
onehot=OneHotEncoder(sparse_output=False)

In [27]:
Embarked=onehot.fit_transform(data[['Embarked']])

In [28]:
Embarked=pd.DataFrame(Embarked,columns=onehot.get_feature_names_out())

In [29]:
Embarked

,Embarked_Cherbourg,Embarked_Queenstown,Embarked_Southampton
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
...,...,...,...
997755,0.0,0.0,1.0
997756,0.0,0.0,1.0
997757,0.0,0.0,1.0
997758,0.0,0.0,1.0


In [30]:
Embarked=Embarked.reset_index(drop=True)

In [31]:
# Joining two dataframes to get final dataset
data=pd.concat([data.drop(columns=['Embarked']),Embarked],axis=1) #axis=1 is for column, we are dropping previous embarked column in dataset.

In [32]:
data

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cherbourg,Embarked_Queenstown,Embarked_Southampton
0,1,1,0,0,0,76,1.0,0.0,0.0
1,0,3,1,0,0,10,0.0,0.0,1.0
2,0,3,1,0,0,12,1.0,0.0,0.0
3,0,3,1,0,0,13,0.0,0.0,1.0
4,0,3,1,0,0,4,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
997755,0,2,1,0,0,26,0.0,0.0,1.0
997756,1,1,0,0,0,102,0.0,0.0,1.0
997757,0,3,1,1,0,8,0.0,0.0,1.0
997758,0,2,1,1,0,0,0.0,0.0,1.0


In [33]:
data.isnull().sum()

Survived                0
Pclass                  0
Sex                     0
SibSp                   0
Parch                   0
Fare                    0
Embarked_Cherbourg      0
Embarked_Queenstown     0
Embarked_Southampton    0
dtype: int64

## Feature Scaling

In [34]:
from sklearn.preprocessing import StandardScaler

In [35]:
scale=StandardScaler()

In [36]:
data.columns

Index(['Survived', 'Pclass', 'Sex', 'SibSp', 'Parch', 'Fare',
       'Embarked_Cherbourg', 'Embarked_Queenstown', 'Embarked_Southampton'],
      dtype='str')

In [37]:
num_cols=['Pclass', 'SibSp', 'Parch', 'Fare'] #we won't include onehotencoded columns and labelencoded columns in scaling as they are already in 0 and 1 format that we have done intendedly. we would scale original numerical data.

In [38]:
data[num_cols]=scale.fit_transform(data[num_cols])

In [39]:
data.head() #standardized data, mean=0 and std=1. Now we can use this data to train our model.

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cherbourg,Embarked_Queenstown,Embarked_Southampton
0,1,-1.568865,0,-0.462644,-0.469267,0.896610,1.0,0.0,0.0
1,0,0.824107,1,-0.462644,-0.469267,-0.479488,0.0,0.0,1.0
2,0,0.824107,1,-0.462644,-0.469267,-0.437788,1.0,0.0,0.0
3,0,0.824107,1,-0.462644,-0.469267,-0.416938,0.0,0.0,1.0
4,0,0.824107,1,-0.462644,-0.469267,-0.604587,1.0,0.0,0.0


In [40]:
import pickle #save encoded and scaled data in pickle file so during deployment we can use it to transform the user input data in same way as we did during training.

In [41]:
with open('label_encoder.pkl','wb') as file:
    pickle.dump(label,file)

In [42]:
with open('onehot_encoder.pkl','wb') as file:
    pickle.dump(onehot,file)

In [43]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scale,file)

## Train Validation Split

In [44]:
X=data.drop(columns=['Survived'])

In [45]:
Y=data['Survived']

In [46]:
X.head()

,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cherbourg,Embarked_Queenstown,Embarked_Southampton
0,-1.568865,0,-0.462644,-0.469267,0.896610,1.0,0.0,0.0
1,0.824107,1,-0.462644,-0.469267,-0.479488,0.0,0.0,1.0
2,0.824107,1,-0.462644,-0.469267,-0.437788,1.0,0.0,0.0
3,0.824107,1,-0.462644,-0.469267,-0.416938,0.0,0.0,1.0
4,0.824107,1,-0.462644,-0.469267,-0.604587,1.0,0.0,0.0


In [47]:
Y.head()

0    1
1    0
2    0
3    0
4    0
Name: Survived, dtype: int64

In [48]:
from sklearn.model_selection import train_test_split

In [49]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42) #20% data for testing and 80% for training, random_state is used to get same split pattern every time we run the code.

In [50]:
X_train,X_valid,Y_train,Y_valid=train_test_split(X_train,Y_train,test_size=0.2,random_state=42) #20% data for validation and 80% for training

In [51]:
X_train.shape

(638566, 8)

In [52]:
X_valid.shape

(159642, 8)

## Model Training
We train a sequential model in DL, meaning input is passed through sequence of layers(input,hidden,output) for learning, and tensorflow is apt for training sequential models. Dense word stands for a fully connected layer: every neuron in that layer is connected to every output from the previous layer.

In [53]:
import tensorflow

In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [55]:
model=Sequential([Dense(128,input_shape=(X_train.shape[1],),activation='relu'),Dense(64,activation='relu'),Dense(32,activation='relu'),Dense(1,activation='sigmoid')]) #First Hidden Layer with 128 neurons, input shape is number of features in training data, activation function is relu. Second Hidden Layer with 64 neurons, activation function is relu. Third Hidden Layer with 32 neurons, activation function is relu. Output Layer with 1 neuron and sigmoid activation for classification. First layer has more neuron to capture more basic features and subsequent layers have fewer neurons to reduce overfitting. Output layer has 1 neuron as we have binary classification problem. We use comma with input shape because it is a tuple syntax to represent dimension like 4, represent one dimension(column) with 4 values. We use relu activation function for hidden layers as it is most commonly used and sigmoid for output layer as we have binary classification problem.

c:\Users\Somya Goyal\Downloads\ANN Project\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,521 (45.00 KB)

 Trainable params: 11,521 (45.00 KB)

 Non-trainable params: 0 (0.00 B)

just 2600 parameters for 10 lakh rows is not a good number so we ned to fix it using hyperparameter tuning that we will do later(number of parameters depend on complexity of the problem not the number of rows, unnecessarily creating more complexity for a simple problem could lead to overfitting or underfitting kind of issues.)

### Model Compilation: Optimizer,Loss and Metrics

In [57]:
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
tensorflow.keras.losses.BinaryFocalCrossentropy


keras.src.losses.losses.BinaryFocalCrossentropy

In [58]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy']) #if we use optimizer as adam directly, it will take default learning rate of 0.001 which is very low and will take more time to converge. So we have defined our own learning rate of 0.01 and binary_crossentropy is used as loss function for binary classification problem. Accuracy is used as metric to evaluate the model performance.

In [59]:
stopping_callback=EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True) #after 5th epoch if validation loss doesn't improve, it will stop training and restore best weights to avoid overfitting.


In [60]:
model.fit(X_train,Y_train,validation_data=(X_valid,Y_valid),epochs=10,callbacks=[stopping_callback]) #training the model with training data and validating with validation data. We have defined 100 epochs but it will stop earlier if validation loss doesn't improve after 5 epochs. We are using less epochs as we are working on CPU and with large data.

Epoch 1/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 117s 6ms/step - accuracy: 0.8560 - loss: 0.3339 - val_accuracy: 0.8599 - val_loss: 0.3188
Epoch 2/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 116s 6ms/step - accuracy: 0.8623 - loss: 0.3182 - val_accuracy: 0.8631 - val_loss: 0.3147
Epoch 3/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 112s 6ms/step - accuracy: 0.8635 - loss: 0.3159 - val_accuracy: 0.8630 - val_loss: 0.3174
Epoch 4/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 110s 5ms/step - accuracy: 0.8641 - loss: 0.3148 - val_accuracy: 0.8638 - val_loss: 0.3122
Epoch 5/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 110s 6ms/step - accuracy: 0.8650 - loss: 0.3135 - val_accuracy: 0.8656 - val_loss: 0.3098
Epoch 6/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 110s 6ms/step - accuracy: 0.8655 - loss: 0.3125 - val_accuracy: 0.8626 - val_loss: 0.3108
Epoch 7/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 115s 6ms/step - accuracy: 0.8657 - loss: 0.3114 - val_accuracy: 0.8647 - val_loss: 0.3108
Epoch 8/10
19956/19956 ━━━━━━━━━━━━━━━━━━━━ 110s 5ms/step - ac

In [61]:
model.save('model.h5') #always save in h5 format so that the weight values and architecture of the model can be saved together. This will help us to load the model later for prediction without needing to recompile or retrain it.

## 1. Machine Learning vs. Deep Learning Workflows

Up to the model training phase, the engineering steps for machine learning and deep learning projects are nearly identical [1]. This includes:
* **Data Gathering:** Acquiring raw data files [1].
* **Data Pre-processing and Cleaning:** Handling missing values, outliers, and incorrect data types [1].
* **Feature Encoding:** Converting categorical variables into numerical values [1].
* **Feature Scaling:** Bringing numerical features to a uniform range to avoid domination problems [2].
* **Train-Test Split:** Dividing the dataset into training and validation/testing sets [2].

### The Core Divergence in Model Training
The fundamental difference between machine learning and deep learning lies in how the models are defined, trained, and optimized [2]:

| Feature | Machine Learning (ML) | Deep Learning (DL) / Artificial Neural Networks (ANN) |
| :--- | :--- | :--- |
| **Algorithm Selection** | You select a pre-defined algorithm (e.g., Logistic Regression, Naive Bayes, Support Vector Machines (SVM), K-Nearest Neighbors (KNN), Decision Trees, Random Forest, AdaBoost, CatBoost, XGBoost) [2, 3]. | You construct a customized, multi-layered architecture from scratch. There is no single named "algorithm" to call [3, 4]. |
| **Model Nature** | Uses specialized mathematical statistical boundaries [2, 3]. | Uses a **Sequential Model**, passing data through a custom sequence of layers (Input $\rightarrow$ Hidden $\rightarrow$ Output) [4, 5]. |
| **Code Implementation** | You instantiate a named classifier and immediately call the `.fit()` method [12]. | You must manually design the layers, node counts, activation functions, optimizers, learning rates, loss functions, and callbacks before calling `.fit()` [4, 12, 13, 15]. |
| **Feature Extraction** | Relies on manual feature engineering and selection [1, 2]. | The initial hidden layers dynamically extract primitive features (e.g., edges, shapes, or basic patterns) and combine them across deeper layers [9]. |

---

## 2. Designing the Sequential Model Architecture

In Keras, the most common model container is the **Sequential model** (`tf.keras.models.Sequential`), where data flows forward layer by layer in a rigid, sequence-by-sequence format [4, 13, 21].

```
  [Inputs: 8 Nodes] ──► [Hidden Layer 1: 64 Nodes] ──► [Hidden Layer 2: 32 Nodes] ──► [Output Layer: 1 Node]
```

### A. The Input Layer and Feature Dimensionality
* **The Input Shape:** The input layer must have exactly the same number of nodes as the number of features in your processed dataset [6, 8, 24].
* **The One-Hot Encoding Expansion:** While the raw Titanic dataset might keep **6 columns** (Passenger Class, Gender, Siblings/Spouse, Parent/Child, Fare, Embarked) [6], performing **one-hot encoding** expands categorical features (like gender and embarkation port), resulting in an actual training shape (`X_train.shape[1]`) of **8 columns** [7, 8].
* **Nodes:** Thus, the input layer must contain exactly **8 input nodes** to match the 8 features [8].

### B. Hidden Layers (Dense Layers)
In Keras, hidden layers are added as **Dense layers** (`tf.keras.layers.Dense`), where every node in the layer is fully connected to every node in the preceding layer [13, 14, 21].
* **First Hidden Layer ($H_1$):** Configured with **64 nodes** [7, 9]. The first hidden layer should generally have a higher node count because it is responsible for extracting all the basic, **primitive features** from the raw inputs [9].
* **Second Hidden Layer ($H_2$):** Configured with **32 nodes** [9]. Subsequent layers combine these primitive features to recognize more complex, high-level patterns [9].

### C. The Output Layer
* **Binary Classification Standard:** Since predicting survival on the Titanic is a **binary classification problem**, the output layer must contain exactly **1 node** [6, 7].
* **Probability Mapping:** This single node outputs a value between $0$ and $1$, representing the probability of survival [15].

---

## 3. Selecting Activation Functions

Every dense layer requires an **Activation Function** to introduce non-linearity into the network, allowing it to learn non-linear decision boundaries [15].

* **Hidden Layers (Use ReLU):** Always use **ReLU (Rectified Linear Unit)** ($\max(0, z)$) inside hidden layers [15, 26]. 
  * *Why?* If you use Sigmoid or Tanh in hidden layers, the activations will saturate for large positive or negative values, causing gradients to shrink to approximately zero during backpropagation [15, 26, 27]. This is the **Vanishing Gradient Problem (VGP)**, which halts all weight updates and stops learning [27]. ReLU preserves gradients in its positive domain, completely preventing VGP [26].
* **Output Layer (Use Sigmoid):** For binary classification, you must use the **Sigmoid** activation function in the output node [15, 28].
  * *Why?* Sigmoid squashes the output into a probability range of $[0, 1]$ [15].

---

## 4. Mathematical Parameter Counting (The Equations)

To understand how deep learning models compile and how memory is allocated, you must be able to calculate the total number of trainable parameters (weights and biases) in the network [10, 30].

### The General Parameter Formula
For any fully connected layer $l$ connected to a preceding layer $l-1$:
$$\text{Parameters} = (\text{Nodes}_{l-1} \times \text{Nodes}_l) + \text{Biases}_l$$
*(where the number of biases in layer $l$ is exactly equal to the number of nodes in layer $l$, $\text{Nodes}_l$)* [11, 31].

### A. Walkthrough of a Basic Toy Network
Let us calculate parameters for a basic network with the following architecture [10]:
* **Inputs:** 2 nodes [10]
* **Hidden Layer 1 ($H_1$):** 3 nodes [10]
* **Hidden Layer 2 ($H_2$):** 3 nodes [10]
* **Output Layer:** 1 node [10]

```
  Layer 1 (Input)      Layer 2 (H1)         Layer 3 (H2)         Layer 4 (Output)
     ( 2 Nodes )  ───►  ( 3 Nodes )  ───►  ( 3 Nodes )  ───►   ( 1 Node )
```

1. **Parameters between Input and $H_1$:**
   $$\text{Weights} = 2 \times 3 = 6 \quad \text{Biases} = 3 \quad \text{Total} = 6 + 3 = 9 \text{ parameters} [11]$$
2. **Parameters between $H_1$ and $H_2$:**
   $$\text{Weights} = 3 \times 3 = 9 \quad \text{Biases} = 3 \quad \text{Total} = 9 + 3 = 12 \text{ parameters} [11]$$
3. **Parameters between $H_2$ and Output:**
   $$\text{Weights} = 3 \times 1 = 3 \quad \text{Biases} = 1 \quad \text{Total} = 3 + 1 = 4 \text{ parameters} [11]$$
4. **Overall Trainable Parameters:**
   $$\text{Total Parameters} = 9 + 12 + 4 = 25 \text{ parameters} [11]$$

---

### B. Calculating Parameters for the Titanic Network
Now, let us calculate the parameter counts shown in `model.summary()` for our actual 8-input, 64-32-1 hidden layer network [7, 8, 30]:

#### 1. First Dense Layer (Input $\rightarrow H_1$)
* **Inputs:** 8 nodes [8, 30]
* **Nodes ($H_1$):** 64 nodes [7, 30]
* **Calculation:**
  $$\text{Parameters} = (8 \times 64) + 64 = 512 + 64 = 576 \text{ parameters} [30, 31]$$

#### 2. Second Dense Layer ($H_1 \rightarrow H_2$)
* **Inputs:** 64 nodes [31]
* **Nodes ($H_2$):** 32 nodes [9, 31]
* **Calculation:**
  $$\text{Parameters} = (64 \times 32) + 32 = 2048 + 32 = 2080 \text{ parameters} [31]$$

#### 3. Output Layer ($H_2 \rightarrow$ Output)
* **Inputs:** 32 nodes [31]
* **Nodes (Output):** 1 node [7, 31]
* **Calculation:**
  $$\text{Parameters} = (32 \times 1) + 1 = 32 + 1 = 33 \text{ parameters} [31, 32]$$

#### 4. Grand Total
$$\text{Total Trainable Parameters} = 576 + 2080 + 33 = 2,689 \text{ parameters} [32]$$

> **Practical Engineering Warning:** While $2,689$ parameters seem large, this network is actually **extremely simple and under-parameterized** for a dataset containing **10 lakh (1 million) rows** [32]. With only 2,689 parameters, you cannot expect high accuracy on such a massive dataset [32]. To train successfully at this scale, you must perform **hyperparameter tuning** to expand the number of layers and hidden nodes [32].

---

## 5. Model Compilation: Optimizers, Loss, and Metrics

Once the layers are constructed, the model must be compiled using `model.compile()`. This step defines the mathematical rules of the training process [33].

### A. Setting a Custom Learning Rate ($\eta$)
* **Predefined Learning Rates:** If you pass the optimizer as a simple string (e.g., `optimizer='adam'`), Keras instantiates the optimizer with a standard predefined learning rate (typically $0.001$) [34, 35].
* **Custom Learning Rates:** To control training speed and stability, you should instantiate the optimizer object manually and specify the learning rate [35].
  * *Example:* If we want to set a learning rate of **$\eta = 0.01$**, we instantiate it using `tf.keras.optimizers.Adam(learning_rate=0.01)` [37].
* **Available Optimizers in Keras:** `AdaDelta`, `AdaGrad`, `Adam`, `AdamW` (designed for GenAI/LLM decoders), `AdaMax`, `Nadam`, `RMSprop`, `SGD`, and `Lion` [36]. For our network, we select **Adam** as the primary optimizer [34, 37].

### B. Defining the Loss Function
* **Binary Cross-Entropy:** Since this is a binary classification problem (survived or did not survive), we use **Binary Cross-Entropy** (`binary_crossentropy`) [34].
* **Categorical Cross-Entropy:** For multi-class classification problems, you would use **Categorical Cross-Entropy** (`categorical_crossentropy`) [37].

### C. Selecting Metrics
* **Classification Standard:** We use **Accuracy** (`metrics=['accuracy']`) to track the percentage of correct predictions [34, 37].
* **Regression Standard:** For regression tasks, you would track **Mean Squared Error (MSE)** or **Mean Absolute Error (MAE)** [18].

---

## 6. Configuring Early Stopping Callbacks

To prevent overfitting and avoid wasting expensive GPU/CPU hours on flatlined training runs, we implement an **Early Stopping Callback** [23, 39, 43].

* **Monitor:** Set to monitor **Validation Loss** (`monitor='val_loss'`) to detect when validation performance starts to deteriorate [39].
* **Patience:** Set to **5** (`patience=5`) [40].
  * *Why?* If validation loss flatlines or rises for a single epoch, we should not stop immediately. Setting patience to 5 forces the watcher to wait for 5 consecutive epochs of no improvement before terminating training [40]. This helps the network escape temporary plateaus or minor gradient hurdles [40].
* **Restore Best Weights:** Set to **True** (`restore_best_weights=True`) [41].
  * *Why?* This ensures that when training is halted, the network automatically discards the sub-optimal weights of the final stagnated epoch and restores the parameters from the epoch that achieved the absolute lowest validation loss [41].

---

## 7. Execution: Training and Saving the Model

* **Batch Size and Training Time:** Training on **10 lakh (1 million) rows** requires processing up to 25,000 batches per epoch [44]. Running this on a standard sequential CPU will take an enormous amount of time [19, 42, 44]. To train rapidly, you should transfer your model to a parallel GPU runtime (such as Google Colab) [19, 42].
* **Epoch Limit:** For demonstration on a CPU, set a low epoch limit (such as **10 epochs**) to save time [42].
* **File Saving Format (H5 over Pickle):** Always save compiled deep learning models using the **H5 file format** (`model.save('model.h5')`) [43]. 
  * *Why?* **Never save deep learning models as pickle files** [43]. The H5 file format is designed to save both the complete model architecture and the exact multi-dimensional weight values in their original high-performance binary format, preventing any numerical degradation [43].

---

## 8. Complete, Runnable Python Implementation

Below is the complete, self-contained Python script to build, compile, train, and save the neural network using TensorFlow/Keras [21, 38, 41, 43]:

```python
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# 1. Access feature dimensionality dynamically from X_train
# X_train.shape[1] is the number of input features (after one-hot encoding, this is 8)
input_dim = X_train.shape[1]  # Dynamic check: input shape representation [24]

# 2. Build the Sequential Network Architecture
model = Sequential([
    # Hidden Layer 1: 64 nodes, with input shape definition and ReLU activation [25, 26]
    Dense(64, input_shape=(input_dim,), activation='relu'), 
    
    # Hidden Layer 2: 32 nodes with ReLU activation [28]
    Dense(32, activation='relu'),
    
    # Output Layer: 1 node with Sigmoid activation for binary classification [28]
    Dense(1, activation='sigmoid')
])

# Display a complete structural breakdown and parameter counts
model.summary()  # Prints parameter tables [30]

# 3. Instantiate Adam Optimizer with Custom Learning Rate
# Setting learning rate to 0.01 instead of default 0.001 for demonstration [37]
custom_opt = tf.keras.optimizers.Adam(learning_rate=0.01)

# 4. Compile the Model with Optimizer, Binary Loss, and Accuracy Metrics
model.compile(
    optimizer=custom_opt,
    loss='binary_crossentropy',  # Ideal for binary classification [34]
    metrics=['accuracy']         # Classification metric tracker [34]
)

# 5. Define Early Stopping Callback with 5-Epoch Patience
stopping_callback = EarlyStopping(
    monitor='val_loss',          # Check validation loss behavior [39]
    patience=5,                  # Wait up to 5 epochs before halting [40]
    restore_best_weights=True    # Revert to optimal parameter states [41]
)

# 6. Fit the Model on 1 Million Rows (Limited to 10 Epochs for CPU runtime)
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,                   # Kept short due to CPU hardware constraints [42]
    callbacks=[stopping_callback] # Register automated stopping watcher [43]
)

# 7. Save the Complete Model and Weights in High-Performance H5 Format
model.save("titanic_nn_model.h5")  # Saves as binary H5 instead of serialization formats [43]
print("Model training completed and saved successfully as 'titanic_nn_model.h5'!")
```

---

## 9. Next Steps in Your Deep Learning Journey

Now that you have configured and trained your first sequential neural network, you are ready to explore:
1. **Hyperparameter Tuning:** Systematically testing different layer depths and neuron widths using frameworks like KerasTuner to optimize network performance [32].
2. **TensorBoard Monitoring:** Logging training and validation performance to visualize loss and accuracy curves in real-time [18].
3. **Advanced Architectures:** Transitioning from simple Dense networks (ANNs) to Convolutional Neural Networks (CNNs) for computer vision or Transformers for Natural Language Processing (NLP) [3].


# Deployment
Render faces issue in multi projects deployment so we will use streamlit cloud which can deploy multi projects. For deployment we must first create UI(website) from where user will enter input and through that input when passed in dataframe and multiple steps will be applied(encoding,scaling etc) will be passed to our model and predictions will be made. use app.py file to make UI always. upload requirement.txt as well so that the deployment environment of streamlit cloud knows which libraries to upload and always upload the necessary files only.

## 1. Part 1: Training Phase Debugging & Stagnation Diagnosis

At the beginning of the session, the instructor addresses a critical bug encountered during the initial training phase of the neural network [15].

### A. The Diagnostic Problem
*   **The Symptoms:** During training on the dataset, the training loss values do not decrease and instead remain frozen [15]. Validation accuracy stays completely stagnant, flatlining around **$61\%$** [15].
*   **The Analysis:** No matter how many epochs run, the network fails to show any incremental improvements or variations in its parameters [15].
*   **The Investigation:** Running a diagnostic check on the data splits reveals the underlying mathematical culprit [15]:
    ```python
    X_train.isnull().sum()
    ```
*   **The Discovery:** The training subset contains a massive volume of **missing values (NaNs)** [15]. 
*   **The Mathematical Cause:** In backpropagation, passing a missing value ($NaN$) through matrix multiplications ($W \cdot X + B$) causes the output to evaluate to $NaN$ [15]. Consequently, the error gradients evaluate to $NaN$, which propagates backward, corrupting all weights and permanently halting any learning [15].

### B. The Crucial Pipeline Rule
The instructor establishes an absolute engineering rule for neural network data cleaning [15]:
> **"Always drop or impute missing values BEFORE performing the train-test split."** [15]

*   **Why?** Removing missing values after the split leads to computational mismatch, alignment crashes, and inconsistent dimensions between target labels ($Y$) and features ($X$) [15].

### C. Implementing the Bug Fix
1.  **Restart Kernel:** Clear the notebook's active memory to reset all variables and configurations [15].
2.  **Clean at the Source:** Drop all missing values directly from the main pandas DataFrame before any splitting occurs [15]:
    ```python
    data = data.dropna()
    ```
3.  **Confirm Cleanliness:** Run a summary check to ensure the dataset has zero null entries remaining [15]:
    ```python
    data.isnull().sum()
    ```
4.  **Free Train-Test Split:** Proceed safely to splitting the cleaned, uniform dataset into training and validation sets [15].

---

## 2. Part 2: Hidden Layer Scaling & Parameter Optimization

With the data-cleaning bug resolved, the instructor shifts focus to scaling up the network's architecture to handle high-volume data [15].

### A. The Data-to-Node Scaling Law
*   **The Baseline Problem:** Initially, the first hidden layer contained only **64 nodes** [15]. 
*   **The Limitation:** When dealing with a massive dataset containing **10 lakh (1 million) rows**, a simple network with a tiny hidden node count is mathematically incapable of capturing complex, high-dimensional non-linear patterns [15].
*   **The Solution:** The hidden node count must scale proportionally with dataset size [15]:
    $$\text{10 Lakh Rows} \implies \text{Node Count should scale up to thousands or lakhs} [15]$$

### B. Refined Model Architecture
The network is redefined in Keras with expanded hidden dimensions [15]:

```
  [Input Layer: 8 Nodes]
            │
            ▼
   [Hidden Layer 1: 128 Nodes] ──► Activation: ReLU
            │
            ▼
   [Hidden Layer 2: 64 Nodes]  ──► Activation: ReLU
            │
            ▼
   [Hidden Layer 3: 64 Nodes]  ──► Activation: ReLU (Wait, sometimes 32 nodes)
            │
            ▼
  [Output Layer: 1 Node]       ──► Activation: Sigmoid (Binary Survival Probability)
```

*   **First Hidden Layer:** Upgraded from $64$ nodes to **$128$ nodes** using **ReLU** activation [15].
*   **Subsequent Layers:** Layer 2 contains **$64$ nodes** (ReLU), followed by Layer 3 containing **$64$ or $32$ nodes** [15].
*   **Output Layer:** Remains a single node with **Sigmoid** activation to output a probability between $0$ and $1$ [15].
*   **Trainable Parameters:** This scaled architecture results in approximately **$11,000$ trainable parameters** [15].

---

## 3. Part 3: CPU vs. GPU Computational Benchmarks

The instructor highlights the hardware limitations of training deep models on standard central processing units (CPUs) [15].

### A. The CPU Training Penalty
*   **The Run Time:** Training the scaled network on **$10$ epochs** over the $10$ lakh rows took exactly **1 hour and 7 minutes** on a standard CPU [15].
*   **The Limitation:** This slow iteration speed is a massive bottleneck. The model's training accuracy improved from **$84\%$ to $85\%$** between epoch 1 and epoch 10, showing that the loss was still steadily decreasing [15].
*   **The Projection:** To achieve optimal convergence (which would require **$100$ epochs** to reach over **$90\%$ accuracy**), a sequential CPU would take **$10$ to $11$ hours** of continuous execution [15].

### B. The Parallel GPU Solution
*   **Why use GPUs?** Unlike CPUs which process tasks sequentially, Graphics Processing Units (GPUs) contain thousands of tiny cores optimized for parallel matrix arithmetic [15].
*   **Recommendation:** For large-scale deep learning (such as handling $10$ lakh rows), always leverage a GPU environment (such as Google Colab's free T4/A100 GPU runtimes) [15]. This cuts training times from hours down to minutes [15].
*   **CPU Alternative (Data Sampling):** If a GPU is unavailable, you must downsample the massive dataset using pandas to run it efficiently on CPU [15]:
    ```python
    data = data.sample(10000)  # Reduces processing volume while preserving patterns
    ```

---

## 4. Part 4: Saving and Loading Serialized Artifacts for Inference

Once model training successfully converges to an accuracy of **$85\%$** (and a validation accuracy of **$85.43\%$**), the model and pre-processing states are serialized [15].

### A. The Complete Inference Artifact Suite
Four separate files must be saved to deploy the model for inference [15]:
1.  **`model.h5`:** The trained Artificial Neural Network (ANN) saved in Keras's binary Hierarchical Data Format [15].
2.  **`scaler.pkl`:** The Standard Scaler saved in a serialized Pickle file [15].
3.  **`label_encoder.pkl`:** The categorical encoder for the binary `gender` column [15].
4.  **`one_hot_encoder.pkl`:** The categorical encoder for the multiclass `embarked` column [15].

### B. Configuring the Prediction Environment
A new, isolated notebook file named **`predict.ipynb`** is created using a **Python 3.11** virtual environment to execute the prediction code [15].

### C. Loading the Artifacts via Python
To perform inference, we load the Keras model and unpickle our pre-processors using **Read Binary (`'rb'`)** mode [15]:

```python
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the trained Keras neural network model
model = load_model('model.h5')

# Load the label encoder for 'gender'
with open('label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)

# Load the one-hot encoder for 'embarked'
with open('one_hot_encoder.pkl', 'rb') as file:
    one_hot_encoder = pickle.load(file)

# Load the standard scaler for continuous features
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)
```

---

## 5. Part 5: End-to-End User Input Pre-Processing Pipeline

Before raw user inputs can be passed into the neural network for prediction, they must undergo the exact same mathematical transformations applied during model training [15].

### A. Formulating the Mock User Input
The instructor sets up a mock dictionary simulating a passenger's details [15]:
```python
user_input = {
    'Pclass': 3,
    'gender': 'female',
    'SibSp': 1,
    'Parch': 1,
    'Fare': 11.0,
    'Embarked': 'Queenstown'
}
```

### B. Sociological Survival Priors (Model Intuitiveness)
The instructor explains the historical priors governing these input features [15]:
*   **Pclass 1 (Wealthiest):** Passengers paid premium fares ($\approx \$76$) and had the highest chance of survival due to lifeboat prioritization [15].
*   **Pclass 3 (Poorest):** Passengers paid low fares ($\approx \$10$ to $\$13$) and had a lower mathematical probability of survival [15].
*   **Embarked (Queenstown - Q):** Pclass 3 passengers frequently boarded from Queenstown, which correlates with lower survival outputs from the model [15].

### C. Step-by-Step Pre-Processing and Inference Sequence
To run predictions on the raw input, the following pipeline is executed [15]:

```
      [Raw User Input Dictionary]
                   │
                   ▼
       [Convert to pd.DataFrame]
                   │
                   ▼
     ┌─────────────┴─────────────┐
     ▼                           ▼
[Label Encode 'gender']   [One-Hot Encode 'Embarked']
     │                           │
     └─────────────┬─────────────┘
                   ▼
      [Concatenate Arrays Back]
                   │
                   ▼
     [Scale Continuous 'Fare' Column]
                   │
                   ▼
       [Pass to model.predict()]
                   │
                   ▼
      [Sigmoid Output: 0.0 to 1.0]
```

1.  **DataFrame Conversion:** Raw dictionary inputs must be wrapped inside a list and converted into a pandas DataFrame to maintain feature headers [15]:
    ```python
    df_user = pd.DataFrame([user_input])
    ```
2.  **Categorical Label Encoding:** Pass the raw `gender` column (e.g., `'female'`) through the loaded `label_encoder` to map it to a numerical binary value [15]:
    ```python
    gender_encoded = label_encoder.transform(df_user['gender'])
    ```
3.  **Multiclass One-Hot Encoding:** Pass the raw `Embarked` column (e.g., `'Queenstown'`) through the `one_hot_encoder` to generate binary dummy columns [15]:
    ```python
    embarked_encoded = one_hot_encoder.transform(df_user[['Embarked']])
    ```
4.  **Concatenation & Alignment:** Concatenate the encoded categorical columns back with numerical columns like `Pclass`, `SibSp`, and `Parch` [15].
5.  **Standard Scaling:** Scale the continuous numeric features (like `Fare`) using the loaded `scaler` to match the model's expected input boundaries [15].
6.  **Inference Prediction:** Pass the final, pre-processed input vector into the network's prediction function [15]:
    ```python
    prediction = model.predict(final_preprocessed_vector)
    ```
7.  **Sigmoid Decision Boundary:** Because the output layer uses a Sigmoid activation, the model returns a probability between $0$ and $1$ [15].
    *   If $\text{Output} \ge 0.5 \implies$ **Survived** [15].
    *   If $\text{Output} < 0.5 \implies$ **Not Survived** [15].


## 1. Bridging Training to Real-Time Inference

During model training, datasets are processed in large batches, using fitting and transformation steps to prepare variables. However, once a deep neural network is deployed, we must build an **inference pipeline** [16]. This pipeline takes a single, raw dictionary of user inputs, applies identical transformations, and passes it to the model for real-time predictions [15, 16].

The core design challenge of inference is maintaining **mathematical alignment** [2, 3]. If our network was trained with specific feature distributions, scaling boundaries, and binary representations, the incoming user inputs must be processed using the *exact same parameters* to prevent model confusion and inaccurate predictions [2].

---

## 2. In-Depth Categorical Encoding Walkthrough

During real-time inference, raw categorical columns (such as gender and embarkation port) must be transformed into numeric formats before they are passed into the input layers of the network [1, 3, 7].

```
  [Raw User Input] ──────► [Label Encoder (transform)] ──────► Gender: 0 or 1
  [Raw Embarked]   ──────► [OneHotEncoder (transform)]  ──────► [0, 1, 0] (3 Columns)
```

### A. Label Encoding for the Gender Column
To handle the binary gender column, we utilize the pre-trained `LabelEncoder` stored during our training phase inside the `label` variable [1]:

*   **The Transform Rule (No Fitting!):**
    When preprocessing incoming data during testing or inference, **we must never use `fit_transform`** [1, 9]. We must strictly use **`transform`** [1, 9].
*   **The Mathematical Reason:**
    *   `fit_transform` calculates and maps new encoding rules from scratch based on the unique elements in the current batch [1, 2].
    *   If we fit a single-row input containing only a female passenger, `fit_transform` might assign her a value of `0` [2, 3]. However, if the next input contains only a male passenger, it would assign him `0` as well, breaking the network's understanding [2].
    *   By using `transform`, we force the encoder to use the pre-established rules from our training set (e.g., if female was mapped to `0` in training, she remains `0` during inference) [2, 3].
*   **Code Implementation:**
    ```python
    # Apply pre-trained label encoder on the real-time 'user' DataFrame
    user['Gender'] = label.transform(user['Gender']) [1, 2]
    ```

### B. One-Hot Encoding for the Embarked Column
The `Embarked` column is multiclass (consisting of Cherbourg, Queenstown, and Southampton) and was trained using a `OneHotEncoder` [3, 5]. We must expand this single column into three binary columns [3, 5]:

1.  **2D Format Requirement:**
    We must pass the embarked feature into our encoder as a **2D array/DataFrame format** (e.g., `user[['Embarked']]`) rather than a flat 1D Series [3].
2.  **Retrieving Feature Columns:**
    Instead of manually naming the newly expanded columns, we extract them dynamically from the pre-trained encoder using `.get_feature_names_out()` [4]. This guarantees that our column names exactly match the spelling and order used during training [4]:
    *   `embarked_Cherbourg` (historically starting from Cherbourg) [5, 19]
    *   `embarked_Queenstown` [5]
    *   `embarked_Southampton` [5]
3.  **Code Implementation:**
    ```python
    # Step 1: Transform embarked into a 2D sparse array
    embarked_sparse = one_hot.transform(user[['Embarked']]) [3]
    
    # Step 2: Convert to a proper Pandas DataFrame with pre-trained feature names
    embark_df = pd.DataFrame(
        embarked_sparse, 
        columns=one_hot.get_feature_names_out()
    ) [4]
    ```

### C. The Column Concatenation Process
Once the embarked column is hot-encoded into a three-column DataFrame, we must merge it back with our main user DataFrame using **column-wise concatenation** (`axis=1`) [5, 6]:

1.  **Dropping the Original Column:** We drop the raw categorical `Embarked` column from the original `user` DataFrame so that it is not duplicated [5, 6].
2.  **Concatenating:** We join the remaining features with the new `embark_df` [5, 6].
3.  **Code Implementation:**
    ```python
    # Concatenate column-wise (axis=1) while dropping the original categorical column
    user = pd.concat([user.drop(columns=['Embarked']), embark_df], axis=1) [5, 6]
    ```
4.  **Resulting Feature Set:** The dataset is now compiled into a standardized format containing exactly 8 numerical features: **Pclass, Gender, SibSp, Parch, Fare, embarked_Cherbourg, embarked_Queenstown, and embarked_Southampton** [7].

---

## 3. Real-Time Numerical Scaling

With all features converted to numbers, we must scale the numerical columns so that no single feature dominates the dot-product calculations [7, 8].

*   **Target Columns to Scale:** `Pclass` (Passenger Class), `SibSp` (Siblings/Spouse), `Parch` (Parent/Child), and `Fare` [8].
*   **The Pre-Trained Scaler:** We use our pre-loaded `scaler` object from our training workspace [8].
*   **Strict Inference Rule:** We use **`.transform()`**, NOT `.fit_transform()` [8, 9]. Fitting here would calculate a new mean and standard deviation from our single user row, causing the scaling to collapse [2, 9].
*   **Mathematical Output:** The scaler centers the values to have a **mean ($\mu$) of 0** and a **standard deviation ($\sigma$) of 1** [9]:
    ```python
    # Scale numeric columns using pre-trained training parameters
    user[['Pclass', 'SibSp', 'Parch', 'Fare']] = scaler.transform(user[['Pclass', 'SibSp', 'Parch', 'Fare']]) [8]
    ```

---

## 4. Executing Predictions & The Generative AI Parallel

Now that our user input is preprocessed, scaled, and formatted, it is ready to be fed directly into our pre-loaded neural network model [9].

```python
# Pass the preprocessed DataFrame into the model's predict function
y_pred = model.predict(user) [10]
```

### A. Probability Outputs in Deep Learning
When we call `model.predict(user)`, the network returns a continuous **probability decimal value** between `0.0` and `1.0` (e.g., `0.22` represents a **22% chance of survival**) [10, 14, 18]. 

The instructor highlights a fundamental deep learning concept: **almost all real-world machine learning and deep learning models output raw probabilities, not final categorical answers** [10, 11, 16].

### B. The Generative AI (LLM) Parallel: Next-Word Prediction
This probability mapping is exactly how massive **Generative AI** models and Transformers (like ChatGPT, LLaMA, or Gemini) work [11].

```
  [Input Text: "The cat sat on a"] ──────► [Transformer LLM] ──────► Probability Distribution
                                                                     * Mat:  0.50 (50%)
                                                                     * Rat:  0.30 (30%)
                                                                     * Bat:  0.20 (20%)
```

1.  **The Goal:** A GPT-like model is trained to predict the next word (token) in a sentence [11].
2.  **The Multi-Class Classification Setup:** Next-word prediction is a multi-class classification problem where the output classes consist of every word in the model's vocabulary (often 32,000 to 100,000+ words) [12].
3.  **The Probability Output:** When given an input prompt like *"The cat sat on a"*, the model outputs a probability value for every word in its vocabulary [11, 12]:
    *   **Mat** is assigned a probability of **0.5 (50%)** [12, 13].
    *   **Rat** is assigned a probability of **0.3 (30%)** [12].
    *   **Bat** is assigned a probability of **0.2 (20%)** [12].
4.  **Selecting the Output:** The model uses this probability distribution to make its next prediction, selecting the word with the highest probability ("Mat") [12, 13].
5.  **Retraining & Backpropagation:** If the model predicts an illogical word with high probability (e.g., assigning a high probability to "rat" instead of "mat"), we must run backpropagation to update the internal weights, refining the network until the probability outputs are correct [13].

---

## 5. Classification Logic and Binary Thresholding

To convert our continuous probability ($y$) into a binary output (Survives vs. Does Not Survive), we apply a **binary threshold of 0.5 (50%)** [14, 15]:

$$\text{Prediction} = \begin{cases} 
\text{Passenger Will Survive} & \text{if } y \ge 0.5 \text{ (50\\%)} \\
\text{Passenger Will Not Survive} & \text{if } y < 0.5 \text{ (50\\%)} 
\end{cases} [15]$$

*   **Accessing the Scalar Value:** Since `model.predict()` returns a nested NumPy array, we access the scalar probability using `y[0][0]` [17, 18].
*   **Code Implementation:**
    ```python
    # Extract the raw probability from the model's prediction array
    probability = y_pred[0][0] [17, 18]
    
    if probability >= 0.5:
        print("The passenger will likely survive the Titanic journey.") [15, 18]
    else:
        print("The passenger won't survive in the Titanic journey.") [15, 18]
    ```

---

## 6. Real-World Scenario Testing (Historical Validation)

To verify if our neural network has learned genuine historical patterns, we test three separate passenger profiles [18, 19, 21]:

### Test Case 1: Low-Income Female Passenger [18]
*   **Profile Details:** Travelled in **Class 3**, female, paid a very low fare, with siblings and parent/child aboard [14, 18].
*   **Model Probability Output:** **20% (0.20)** survival chance [14, 18].
*   **Result:** Classified as **"Won't Survive"** [18].
*   **Historical Realism:** This matches the historical event depicted in the movie, where lower-class passengers on lower decks had significantly lower survival rates [14].

### Test Case 2: High-Income Female Passenger [18, 19, 20]
*   **Profile Details:** Travelled in **Class 1**, female, paid a high fare of **$76**, embarked from Cherbourg [18, 19, 20].
*   **Model Probability Output:** **99.59% (0.9959)** survival chance [20].
*   **Result:** Classified as **"Likely Survive"** [20].
*   **Historical Realism:** In the historical event, first-class female passengers were given absolute priority during life-boat boarding [20, 21].

### Test Case 3: High-Income Male Passenger [21]
*   **Profile Details:** We change only the gender of Test Case 2 to **Male** while keeping him in **Class 1**, paying a high fare of **$76**, and starting from Cherbourg [21].
*   **Model Probability Output:** **5.53% (0.0553)** survival chance [21].
*   **Result:** Classified as **"Won't Survive"** [21].
*   **Historical Realism:** This demonstrates that the model is working exceptionally well [21, 22]. Despite being rich and traveling in first class, males were given lowest priority for lifeboat boarding, leading to low survival rates [21].

---

## 7. Model Scaling and Hyperparameter Options

*   **Current Model Accuracy:** Currently achieves **85% accuracy** on validation sets [16, 22].
*   **Scaling Up for Higher Accuracy:** 
    1.  **More Training Epochs:** Running the model for 100 epochs would likely push the validation accuracy past 90%, but this requires parallel GPU resources to complete in a reasonable time [17, 22].
    2.  **Deeper Network Architectures:** Adding multiple hidden layers containing a carefully designed number of perceptrons to handle complex, high-dimensional patterns can scale performance up to **95% accuracy** [17, 22].
*   **Next Steps:** In the next session, we will wrap this exact python preprocessing, scaling, and prediction logic inside a **Streamlit** user interface, creating interactive sliders, select boxes, and input blocks to build a fully functional web application [15, 16, 22].

---

